# SprayCount — RF-DETR Medium inference over Cloudflare Realtime

This notebook is the GPU half of the SprayCount pipeline. **It does inference and nothing else.**

```
edge-worker ──publish cam-01/cam-02──► Cloudflare Realtime
     │                                        │
     └─POST /api/inference/register───┐       │ (raw tracks)
                                      ▼       ▼
                                  Next.js   THIS NOTEBOOK
                                      ▲       │  GET  /api/inference/source
                                      │       │  POST /api/inference/detections
                                      └───────┤  POST /api/inference/register
                                              │
                                              └─publish annotated──► Cloudflare
                                                                          │
                                       browser ◄─subscribe processed──────┘
```

The flow:

1. The edge worker publishes each camera into a Cloudflare Realtime session and registers it with the Next.js app.
2. This notebook discovers that session via `GET /api/inference/source` — nothing is pasted by hand.
3. One shared **RF-DETR Medium** checkpoint runs on the newest frame from each camera.
4. Annotated video is published back into one processed Cloudflare session, one track per camera, and registered with Next.js so the dashboard finds it automatically.
5. Raw per-frame detections are POSTed to `/api/inference/detections` in frame-normalized coordinates.

**All counting logic lives in Next.js**, not here: spindle-boundary filtering, boundary normalization, `DETECTION_INTERVAL` windowing with `max()`, the `MAX_HOTWHEELS` plausibility filter, visit segmentation, and the FIFO that gives both cameras' views of one spindle the same `spindle_pass_id`. Those are tuned by editing `.env` and restarting the `nextjs` container — this notebook keeps running.

Documentation:

- https://rfdetr.roboflow.com/latest/reference/rfdetr/
- https://rfdetr.roboflow.com/latest/reference/medium/


## 1. Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU** before running the notebook.


In [ ]:
%pip install -q -U rfdetr supervision av aiortc aiohttp

In [ ]:
import asyncio
import getpass
import json
import re
import threading
import time
import uuid
import warnings
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import aiohttp
import av
import numpy as np
import torch

from PIL import Image, ImageDraw, ImageFont
from rfdetr import RFDETR, RFDETRMedium

from aiortc import (
    MediaStreamTrack,
    RTCConfiguration,
    RTCIceServer,
    RTCPeerConnection,
    RTCRtpReceiver,
    RTCSessionDescription,
    VideoStreamTrack,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab choose Runtime → Change runtime type → GPU."
    )

DEVICE = 0
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__)


## 2. Cloudflare App Secret

Only the **secret** is entered by hand — it is hidden while typing and is never sent to the Next.js app.

The **App ID** is fetched from Next.js in section 3, so this notebook always lands on the same Cloudflare application the edge worker is publishing to. Use the secret belonging to the `CF_APP_ID` in the project's `.env`.


In [ ]:
APP_SECRET = getpass.getpass("Cloudflare Realtime App Secret: ").strip()
if not APP_SECRET:
    raise ValueError("App Secret cannot be empty.")

# APP_ID is NOT entered here. It is read from the Next.js app in section 3, so
# this notebook is guaranteed to be on the same Cloudflare application as the
# edge worker publishing the cameras. A hardcoded default here was a genuine
# trap: pressing Enter accepted an App ID that no longer matched .env, and the
# mismatch only surfaced much later as an opaque "track not found".
APP_ID: Optional[str] = None

print("App Secret captured. App ID will be discovered from Next.js.")


## 2b. Connect to the SprayCount Next.js app

Colab runs in Google's cloud and cannot reach `localhost:3000`. Expose the app with a tunnel on the machine running Docker:

```bash
cloudflared tunnel --url http://localhost:3000
# → https://<random>.trycloudflare.com
```

Paste that URL below. It changes every time the tunnel restarts, which is why it is entered here rather than hardcoded.

`INFERENCE_API_KEY` must match the value in the project's `.env`. The tunnel is publicly addressable, so this shared secret is the only thing standing between the open internet and the count pipeline.

Colab makes two calls to the app:

| Call | Purpose |
|------|---------|
| `GET /api/inference/source` | Discover which Cloudflare session the edge worker is publishing |
| `POST /api/inference/detections` | Stream raw per-frame detections back for sampling |

**This notebook does inference only.** Spindle-boundary filtering, interval sampling, `max()`, the `MAX_HOTWHEELS` plausibility check, visit segmentation and cross-camera pairing all happen server-side, so those can be tuned by editing `.env` and restarting the `nextjs` container while this notebook keeps running.


In [ ]:
NEXTJS_BASE_URL = input(
    "Next.js base URL (e.g. https://xxxx.trycloudflare.com): "
).strip().rstrip("/")

INFERENCE_API_KEY = getpass.getpass("INFERENCE_API_KEY (from the project .env): ").strip()

# How often buffered frames are shipped to Next.js. RF-DETR Medium on a T4 with
# the shared model lock runs roughly 10-15 fps across all cameras, so half a
# second batches ~5-8 frames per request instead of one request per frame.
REPORT_FLUSH_SECONDS = 0.5

# Bounds memory if Next.js becomes unreachable. Dropping the oldest frames is
# correct here: the server takes max() over a 2 s window, so stale frames from
# an outage are worthless anyway.
REPORT_BUFFER_MAXLEN = 200

if not NEXTJS_BASE_URL.startswith(("http://", "https://")):
    raise ValueError("NEXTJS_BASE_URL must start with http:// or https://")
if not INFERENCE_API_KEY:
    raise ValueError("INFERENCE_API_KEY cannot be empty.")

INFERENCE_HEADERS = {"x-inference-key": INFERENCE_API_KEY}

print("Next.js base URL:", NEXTJS_BASE_URL)


async def check_nextjs_reachable() -> None:
    """Fail fast and loudly rather than silently dropping every count later."""
    url = f"{NEXTJS_BASE_URL}/api/inference/source"
    timeout = aiohttp.ClientTimeout(total=10)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.get(url, headers=INFERENCE_HEADERS) as response:
            body = await response.text()
            if response.status == 401:
                raise RuntimeError(
                    "Next.js rejected INFERENCE_API_KEY. Confirm it matches the "
                    "value in the project's .env."
                )
            if response.status == 404:
                print(
                    "Reached Next.js, but no source sessions are registered yet.\n"
                    "Start the edge worker (docker compose up -d edge-worker) and "
                    "confirm its logs show 'Source sessions registered'."
                )
                return
            if response.status >= 300:
                raise RuntimeError(f"HTTP {response.status} from {url}: {body[:300]}")
            print("Next.js reachable; source sessions are registered.")


await check_nextjs_reachable()


## 3. Discover the source cameras

The edge worker publishes each camera into a Cloudflare Realtime session and registers that session with Next.js on a 15 s heartbeat. This cell reads it back, so nothing has to be copied by hand.

Each entry carries a canonical `cameraId` (`CAM-01`, `CAM-02`). That id — not the display name — is the key everything downstream is joined on, including the counts posted back to Next.js.

If Next.js is unreachable, set `MANUAL_SOURCE_COORDINATES` to a JSON or ENV block and re-run; the fallback parser is retained for that case.


In [ ]:
# Leave empty to auto-discover from Next.js. Set to a JSON or ENV block only if
# Next.js is unreachable and the session has to be supplied by hand.
MANUAL_SOURCE_COORDINATES = ""


def parse_source_coordinates(raw: str) -> dict[str, Any]:
    """Fallback parser for a hand-supplied JSON or ENV block.

    Each camera carries its OWN sessionId, not one shared session: cameras
    never share a Cloudflare connection (see publish_cloudflare_video's
    docstring in section 7 for why bundling multiple tracks onto one
    connection does not work).
    """
    raw = raw.strip()
    if not raw:
        raise ValueError("No source coordinates were provided.")

    app_id = ""

    if raw.startswith("{"):
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON source coordinates: {exc}") from exc

        app_id = str(data.get("appId", "")).strip()
        cameras_raw = data.get("cameras", [])
        if not isinstance(cameras_raw, list):
            raise ValueError("JSON field 'cameras' must be a list.")

        cameras = []
        for index, item in enumerate(cameras_raw, start=1):
            if not isinstance(item, dict):
                raise ValueError(f"cameras[{index - 1}] must be an object.")
            cameras.append(
                {
                    "name": str(item.get("name") or f"Camera {index}").strip(),
                    "trackName": str(item.get("trackName", "")).strip(),
                    "cameraId": str(item.get("cameraId", "")).strip(),
                    "sessionId": str(item.get("sessionId", "")).strip(),
                }
            )
    else:
        env: dict[str, str] = {}
        for raw_line in raw.splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                raise ValueError(f"Invalid ENV line: {raw_line!r}")
            key, value = line.split("=", 1)
            env[key.strip()] = value.strip()

        app_id = env.get("SOURCE_APP_ID", "").strip()
        try:
            camera_count = int(env.get("SOURCE_CAMERA_COUNT", "0"))
        except ValueError as exc:
            raise ValueError("SOURCE_CAMERA_COUNT must be an integer.") from exc

        cameras = []
        for index in range(1, camera_count + 1):
            prefix = f"SOURCE_CAMERA_{index}_"
            cameras.append(
                {
                    "name": env.get(f"{prefix}NAME", f"Camera {index}").strip(),
                    "trackName": env.get(f"{prefix}TRACK_NAME", "").strip(),
                    "cameraId": env.get(f"{prefix}CAMERA_ID", "").strip(),
                    "sessionId": env.get(f"{prefix}SESSION_ID", "").strip(),
                }
            )

    if not cameras:
        raise ValueError("No camera entries were found.")

    return {"cameras": cameras, "appId": app_id}


async def fetch_source_coordinates() -> dict[str, Any]:
    """Read the source sessions the edge worker registered with Next.js."""
    url = f"{NEXTJS_BASE_URL}/api/inference/source"
    timeout = aiohttp.ClientTimeout(total=15)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.get(url, headers=INFERENCE_HEADERS) as response:
            body = await response.text()
            if response.status == 404:
                raise RuntimeError(
                    "No source sessions are registered. Start the edge worker and "
                    "wait for its log line 'Source sessions registered'."
                )
            if response.status >= 300:
                raise RuntimeError(f"HTTP {response.status} from {url}: {body[:300]}")
            data = json.loads(body)

    stale_cameras = [c["cameraId"] for c in data.get("cameras", []) if c.get("stale")]
    if stale_cameras:
        # The Cloudflare session is very likely already gone, which would
        # otherwise surface as an opaque Cloudflare error much later.
        raise RuntimeError(
            f"Source session(s) for {stale_cameras} are stale. "
            "Is the edge worker still running?"
        )

    return data


def normalize_cameras(config: dict[str, Any]) -> dict[str, Any]:
    cameras = []
    seen_track_names: set[str] = set()
    for index, camera in enumerate(config.get("cameras", []), start=1):
        track_name = str(camera.get("trackName", "")).strip()
        if not track_name:
            raise ValueError(f"Camera {index} is missing trackName.")
        if track_name in seen_track_names:
            raise ValueError(f"Duplicate source trackName: {track_name}")
        seen_track_names.add(track_name)

        # Hand-supplied blocks predate cameraId; the edge worker's track names
        # are the lowercased camera ids, so this recovers it.
        camera_id = str(camera.get("cameraId", "")).strip() or track_name.upper()

        session_id = str(camera.get("sessionId", "")).strip()
        if not session_id:
            raise ValueError(
                f"Camera {camera_id} is missing sessionId. Each camera needs its "
                "own Cloudflare session — see MANUAL_SOURCE_COORDINATES above."
            )

        cameras.append(
            {
                "cameraId": camera_id,
                "trackName": track_name,
                "sessionId": session_id,
                "name": str(camera.get("name") or camera_id).strip(),
            }
        )

    if not cameras:
        raise ValueError("No camera entries were found.")
    return {
        "cameras": cameras,
        "appId": str(config.get("appId") or "").strip(),
    }


# Set this only when supplying source coordinates by hand, since the App ID
# normally arrives from Next.js alongside the sessions.
MANUAL_APP_ID = ""


if MANUAL_SOURCE_COORDINATES.strip():
    SOURCE_CONFIG = normalize_cameras(
        parse_source_coordinates(MANUAL_SOURCE_COORDINATES)
    )
    print("Using MANUAL_SOURCE_COORDINATES.")
else:
    SOURCE_CONFIG = normalize_cameras(await fetch_source_coordinates())
    print("Discovered the source sessions from Next.js.")

SOURCE_CAMERAS = SOURCE_CONFIG["cameras"]

# Taking the App ID from the same response as the sessions guarantees this
# notebook and the edge worker are on one Cloudflare application. Subscribing
# with a mismatched App ID fails late and unhelpfully.
APP_ID = MANUAL_APP_ID.strip() or (SOURCE_CONFIG.get("appId") or "")
if not APP_ID:
    raise ValueError(
        "Next.js did not return a Cloudflare App ID. Set CF_APP_ID in the "
        "project's .env and restart the nextjs container, or set MANUAL_APP_ID "
        "above."
    )

print("Cloudflare App ID:", APP_ID)
print("Source cameras (each on its own Cloudflare session):")
for camera in SOURCE_CAMERAS:
    print(f"- {camera['cameraId']} ({camera['name']}): session {camera['sessionId']}, track {camera['trackName']}")


## 4. Configure and load the trained RF-DETR Medium checkpoint

1. Put the trained checkpoint in Google Drive. The recommended inference file is normally `checkpoint_best_total.pth`.
2. Set `CHECKPOINT_PATH` to its exact Drive path.
3. Keep `TRUST_CHECKPOINT=True` only for a checkpoint you created or fully trust.
4. Leave `INFERENCE_SHAPE=None` to use the resolution stored by the model. A manual shape must satisfy RF-DETR's architecture divisibility constraints.
5. Leave `TARGET_CLASS_NAMES` empty to retain every checkpoint class.

The loader first uses `RFDETR.from_checkpoint()` so the architecture and class count are inferred from the checkpoint. If an older checkpoint lacks enough metadata, it falls back to `RFDETRMedium(pretrain_weights=...)`.


In [ ]:
# Mount Google Drive when the checkpoint is stored there.
USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = "/content/drive"

# Change this to your trained RF-DETR Medium checkpoint.
CHECKPOINT_PATH = (
    "/content/esp-count.pth"
)

# Only enable for a checkpoint you created or fully trust.
TRUST_CHECKPOINT = True

CONFIDENCE = 0.35
MAX_DETECTIONS = 100
TARGET_CLASS_NAMES: list[str] = []  # Example: ["hot-wheels"]

# None uses the resolution reconstructed from the checkpoint.
# Example override for RF-DETR Medium: (576, 576)
INFERENCE_SHAPE: Optional[tuple[int, int]] = None

# RF-DETR's optional inference optimization. compile=False is more compatible
# with Colab and WebRTC's dynamic runtime. FP16 is suitable for CUDA inference.
OPTIMIZE_FOR_INFERENCE = True
OPTIMIZE_COMPILE = False
OPTIMIZE_INPLACE = True
USE_HALF_PRECISION = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

checkpoint_file = Path(CHECKPOINT_PATH).expanduser()
if not checkpoint_file.is_file():
    raise FileNotFoundError(
        f"RF-DETR checkpoint was not found: {checkpoint_file}"
    )

print(f"Loading RF-DETR checkpoint: {checkpoint_file}")

try:
    rf_model = RFDETR.from_checkpoint(
        checkpoint_file,
        trust_checkpoint=TRUST_CHECKPOINT,
    )
except ValueError as auto_detect_error:
    warnings.warn(
        "The checkpoint architecture could not be inferred automatically. "
        "Falling back to RFDETRMedium(pretrain_weights=...). "
        f"Original error: {auto_detect_error}"
    )
    rf_model = RFDETRMedium(
        pretrain_weights=str(checkpoint_file),
        trust_checkpoint=TRUST_CHECKPOINT,
    )

loaded_model_class = type(rf_model).__name__
if "Medium" not in loaded_model_class:
    warnings.warn(
        f"The checkpoint reconstructed {loaded_model_class}, not RFDETRMedium. "
        "Verify that CHECKPOINT_PATH points to the intended Medium checkpoint."
    )

MODEL_CLASS_NAMES = [str(name) for name in rf_model.class_names]
if not MODEL_CLASS_NAMES:
    raise RuntimeError("The checkpoint did not expose any class names.")

available_names = {name.casefold(): name for name in MODEL_CLASS_NAMES}
missing_target_names = [
    name for name in TARGET_CLASS_NAMES
    if name.casefold() not in available_names
]
if missing_target_names:
    raise ValueError(
        f"Unknown TARGET_CLASS_NAMES: {missing_target_names}. "
        f"Available classes: {MODEL_CLASS_NAMES}"
    )

NORMALIZED_TARGET_CLASS_NAMES: Optional[set[str]] = (
    {name.casefold() for name in TARGET_CLASS_NAMES}
    if TARGET_CLASS_NAMES
    else None
)

if OPTIMIZE_FOR_INFERENCE:
    rf_model.inference(
        compile=OPTIMIZE_COMPILE,
        batch_size=1,
        dtype=torch.float16 if USE_HALF_PRECISION else torch.float32,
        inplace=OPTIMIZE_INPLACE,
    )

# Warm up once so checkpoint initialization does not happen during signaling.
warmup_height, warmup_width = INFERENCE_SHAPE or (576, 576)
warmup_image = np.zeros(
    (warmup_height, warmup_width, 3),
    dtype=np.uint8,
)
warmup_kwargs: dict[str, Any] = {
    "threshold": CONFIDENCE,
    "include_source_image": False,
}
if INFERENCE_SHAPE is not None:
    warmup_kwargs["shape"] = INFERENCE_SHAPE

_ = rf_model.predict(warmup_image, **warmup_kwargs)

print("RF-DETR checkpoint loaded.")
print("Reconstructed class:", loaded_model_class)
print("Checkpoint:", checkpoint_file.name)
print("Classes:", MODEL_CLASS_NAMES)
print("Class filter:", TARGET_CLASS_NAMES or "all checkpoint classes")


## 5. Cloudflare Realtime API and WebRTC helpers

In [ ]:
class CloudflareRealtimeAPI:
    def __init__(
        self,
        app_id: str,
        app_secret: str,
        base_url: str = "https://rtc.live.cloudflare.com/v1",
    ):
        self.app_id = app_id
        self.app_secret = app_secret
        self.prefix = f"{base_url}/apps/{app_id}"
        self.session_id: Optional[str] = None

    async def _request(
        self,
        path: str,
        body: dict[str, Any],
        method: str = "POST",
    ) -> dict[str, Any]:
        url = f"{self.prefix}{path}"
        headers = {
            "content-type": "application/json",
            "authorization": f"Bearer {self.app_secret}",
        }
        timeout = aiohttp.ClientTimeout(total=60)
        async with aiohttp.ClientSession(timeout=timeout) as session:
            async with session.request(
                method,
                url,
                headers=headers,
                json=body,
            ) as response:
                text = await response.text()
                try:
                    result = json.loads(text)
                except Exception as exc:
                    raise RuntimeError(
                        f"Cloudflare returned HTTP {response.status}: {text[:500]}"
                    ) from exc

                if response.status < 200 or response.status >= 300:
                    raise RuntimeError(
                        f"Cloudflare returned HTTP {response.status}: {result}"
                    )
                self._check_errors(result)
                return result

    @staticmethod
    def _check_errors(result: dict[str, Any]) -> None:
        if result.get("errorCode"):
            raise RuntimeError(
                f"{result.get('errorCode')}: {result.get('errorDescription')}"
            )
        for index, track in enumerate(result.get("tracks", [])):
            if track.get("errorCode"):
                raise RuntimeError(
                    f"tracks[{index}] {track.get('errorCode')}: "
                    f"{track.get('errorDescription')}"
                )

    async def new_session(self, offer_sdp: str) -> dict[str, Any]:
        result = await self._request(
            "/sessions/new",
            {
                "sessionDescription": {
                    "type": "offer",
                    "sdp": offer_sdp,
                }
            },
        )
        self.session_id = result["sessionId"]
        return result

    async def new_tracks(
        self,
        tracks: list[dict[str, Any]],
        offer_sdp: Optional[str] = None,
    ) -> dict[str, Any]:
        if not self.session_id:
            raise RuntimeError("Cloudflare session has not been created.")
        body: dict[str, Any] = {"tracks": tracks}
        if offer_sdp is not None:
            body["sessionDescription"] = {
                "type": "offer",
                "sdp": offer_sdp,
            }
        return await self._request(
            f"/sessions/{self.session_id}/tracks/new",
            body,
        )

    async def renegotiate(self, answer_sdp: str) -> dict[str, Any]:
        if not self.session_id:
            raise RuntimeError("Cloudflare session has not been created.")
        return await self._request(
            f"/sessions/{self.session_id}/renegotiate",
            {
                "sessionDescription": {
                    "type": "answer",
                    "sdp": answer_sdp,
                }
            },
            method="PUT",
        )


def rtc_configuration() -> RTCConfiguration:
    return RTCConfiguration(
        iceServers=[RTCIceServer(urls=["stun:stun.cloudflare.com:3478"])]
    )


def force_vp8(transceiver) -> None:
    """Restrict a video transceiver to VP8 to avoid H.264 decoder stalls."""
    capabilities = RTCRtpReceiver.getCapabilities("video")
    vp8_codecs = [
        codec
        for codec in capabilities.codecs
        if codec.mimeType.casefold() == "video/vp8"
    ]
    if not vp8_codecs:
        raise RuntimeError("This aiortc build does not expose the VP8 codec.")
    transceiver.setCodecPreferences(vp8_codecs)


def force_all_video_transceivers_to_vp8(pc: RTCPeerConnection) -> None:
    for transceiver in pc.getTransceivers():
        if transceiver.kind == "video":
            force_vp8(transceiver)


async def wait_for_ice_gathering(
    pc: RTCPeerConnection,
    timeout_seconds: float = 20.0,
    stage: str = "ICE gathering",
) -> None:
    if pc.iceGatheringState == "complete":
        return

    event = asyncio.Event()

    @pc.on("icegatheringstatechange")
    async def _on_state_change() -> None:
        if pc.iceGatheringState == "complete":
            event.set()

    try:
        await asyncio.wait_for(event.wait(), timeout=timeout_seconds)
    except TimeoutError as exc:
        raise TimeoutError(
            f"{stage} timed out after {timeout_seconds:.0f}s "
            f"(iceGatheringState={pc.iceGatheringState})."
        ) from exc


async def set_complete_local_description(
    pc: RTCPeerConnection,
    description: RTCSessionDescription,
    stage: str = "local description",
) -> RTCSessionDescription:
    await pc.setLocalDescription(description)
    await wait_for_ice_gathering(pc, stage=f"{stage}: ICE gathering")
    if pc.localDescription is None:
        raise RuntimeError(f"{stage}: PeerConnection has no local description.")
    return pc.localDescription


async def wait_for_connection(
    pc: RTCPeerConnection,
    timeout_seconds: float = 60.0,
    stage: str = "WebRTC connection",
) -> None:
    def is_connected() -> bool:
        return (
            pc.connectionState == "connected"
            or pc.iceConnectionState in {"connected", "completed"}
        )

    if is_connected():
        return

    event = asyncio.Event()

    def report() -> None:
        print(
            f"{stage}: connection={pc.connectionState}, "
            f"ICE={pc.iceConnectionState}, signaling={pc.signalingState}"
        )
        if is_connected() or pc.connectionState in {"failed", "closed"}:
            event.set()
        elif pc.iceConnectionState in {"failed", "closed"}:
            event.set()

    @pc.on("connectionstatechange")
    async def _on_connection_state() -> None:
        report()

    @pc.on("iceconnectionstatechange")
    async def _on_ice_state() -> None:
        report()

    try:
        await asyncio.wait_for(event.wait(), timeout=timeout_seconds)
    except TimeoutError as exc:
        raise TimeoutError(
            f"{stage} timed out after {timeout_seconds:.0f}s "
            f"(connection={pc.connectionState}, ICE={pc.iceConnectionState}, "
            f"signaling={pc.signalingState})."
        ) from exc

    if not is_connected():
        raise RuntimeError(
            f"{stage} failed "
            f"(connection={pc.connectionState}, ICE={pc.iceConnectionState})."
        )


def description_from_result(result: dict[str, Any]) -> RTCSessionDescription:
    description = result["sessionDescription"]
    return RTCSessionDescription(
        sdp=description["sdp"],
        type=description["type"],
    )


## 6. Low-latency multi-camera RF-DETR Medium tracks

In [ ]:
@dataclass
class CameraPipelineStats:
    camera_id: str
    camera_name: str
    source_track_name: str
    output_track_name: str
    received_frames: int = 0
    processed_frames: int = 0
    last_inference_ms: float = 0.0
    last_object_count: int = 0
    started_at: float = 0.0

    @property
    def uptime_seconds(self) -> float:
        return max(0.0, time.monotonic() - self.started_at)

    @property
    def processed_fps(self) -> float:
        if self.uptime_seconds <= 0:
            return 0.0
        return self.processed_frames / self.uptime_seconds


class LatestFrameBuffer:
    """Continuously consumes a source and retains only its newest frame."""

    def __init__(self, source: MediaStreamTrack, stats: CameraPipelineStats):
        self.source = source
        self.stats = stats
        self._latest: Optional[av.VideoFrame] = None
        self._sequence = 0
        self._event = asyncio.Event()
        self._exception: Optional[BaseException] = None
        self._task = asyncio.create_task(self._reader())

    async def _reader(self) -> None:
        try:
            while True:
                frame = await self.source.recv()
                self.stats.received_frames += 1
                self._latest = frame
                self._sequence += 1
                self._event.set()
        except BaseException as exc:
            self._exception = exc
            self._event.set()

    async def newest_after(
        self,
        previous_sequence: int,
    ) -> tuple[int, av.VideoFrame]:
        while True:
            if self._exception is not None:
                raise self._exception
            if self._latest is not None and self._sequence > previous_sequence:
                return self._sequence, self._latest
            self._event.clear()
            await self._event.wait()

    async def wait_for_first_frame(
        self,
        timeout_seconds: float = 60.0,
    ) -> av.VideoFrame:
        try:
            _, frame = await asyncio.wait_for(
                self.newest_after(0),
                timeout=timeout_seconds,
            )
            return frame
        except TimeoutError as exc:
            raise TimeoutError(
                f"No decodable video frame arrived for "
                f"{self.stats.camera_name} / {self.stats.source_track_name} "
                f"within {timeout_seconds:.0f}s. "
                "Keep the edge worker publishing and use VP8."
            ) from exc

    async def close(self) -> None:
        self._task.cancel()
        try:
            await self._task
        except BaseException:
            pass


class RFDETRStreamingEngine:
    """A single shared RF-DETR model used safely by all camera tracks.

    Runs inference and annotation only. It emits every detection it finds, in
    frame-normalized coordinates, and makes no judgement about which toys
    "belong" to the spindle — that decision, along with all interval sampling,
    lives server-side in Next.js so it can be tuned without touching this
    notebook.
    """

    def __init__(
        self,
        model: RFDETR,
        confidence: float,
        max_detections: int,
        target_class_names: Optional[set[str]],
        inference_shape: Optional[tuple[int, int]],
    ):
        self.model = model
        self.confidence = confidence
        self.max_detections = max_detections
        self.target_class_names = target_class_names
        self.inference_shape = inference_shape
        self._lock = threading.Lock()

    def process(
        self,
        rgb: np.ndarray,
        camera_name: str,
    ) -> tuple[np.ndarray, float, int, list[dict[str, Any]]]:
        # RF-DETR expects NumPy images in RGB channel order. It returns boxes in
        # the source image coordinate system, so annotation can stay full-size.
        predict_kwargs: dict[str, Any] = {
            "threshold": self.confidence,
            "include_source_image": False,
        }
        if self.inference_shape is not None:
            predict_kwargs["shape"] = self.inference_shape

        # A shared model is serialized because simultaneous predict() calls on
        # one instance are not assumed to be thread-safe.
        with self._lock:
            started_at = time.perf_counter()
            detections = self.model.predict(
                np.ascontiguousarray(rgb),
                **predict_kwargs,
            )
            latency_ms = (time.perf_counter() - started_at) * 1000.0

        boxes = np.asarray(detections.xyxy, dtype=np.float32)
        detection_count = len(boxes)

        if detections.confidence is None:
            confidences = np.ones(detection_count, dtype=np.float32)
        else:
            confidences = np.asarray(
                detections.confidence,
                dtype=np.float32,
            )

        if detections.class_id is None:
            class_ids = np.full(detection_count, -1, dtype=np.int32)
        else:
            class_ids = np.asarray(detections.class_id, dtype=np.int32)

        raw_class_names = detections.data.get("class_name")
        if raw_class_names is not None and len(raw_class_names) == detection_count:
            class_names = np.asarray(
                [str(name) for name in raw_class_names],
                dtype=object,
            )
        else:
            class_names = np.asarray(
                [
                    MODEL_CLASS_NAMES[class_id]
                    if 0 <= class_id < len(MODEL_CLASS_NAMES)
                    else f"class-{class_id}"
                    for class_id in class_ids
                ],
                dtype=object,
            )

        if self.target_class_names is not None and detection_count:
            class_mask = np.asarray(
                [
                    str(name).casefold() in self.target_class_names
                    for name in class_names
                ],
                dtype=bool,
            )
            boxes = boxes[class_mask]
            confidences = confidences[class_mask]
            class_ids = class_ids[class_mask]
            class_names = class_names[class_mask]

        if len(boxes) > self.max_detections:
            selected = np.argsort(confidences)[::-1][: self.max_detections]
            boxes = boxes[selected]
            confidences = confidences[selected]
            class_ids = class_ids[selected]
            class_names = class_names[selected]

        object_count = len(boxes)
        image = Image.fromarray(rgb.copy())
        draw = ImageDraw.Draw(image)
        font = ImageFont.load_default()
        image_width, image_height = image.size

        # Normalized by frame dimensions so the server never has to know the
        # source resolution, and a camera swap cannot silently shift every box.
        payload: list[dict[str, Any]] = []

        for box, confidence, class_name in zip(
            boxes,
            confidences,
            class_names,
        ):
            x1, y1, x2, y2 = [float(value) for value in box]

            payload.append(
                {
                    "cls": str(class_name),
                    "conf": round(float(confidence), 4),
                    "box": [
                        round(x1 / image_width, 5),
                        round(y1 / image_height, 5),
                        round(x2 / image_width, 5),
                        round(y2 / image_height, 5),
                    ],
                }
            )

            x1 = int(max(0, min(image_width - 1, round(x1))))
            y1 = int(max(0, min(image_height - 1, round(y1))))
            x2 = int(max(0, min(image_width - 1, round(x2))))
            y2 = int(max(0, min(image_height - 1, round(y2))))

            # Spindles get their own colour so the operator can see the region
            # the server is measuring toys against.
            is_spindle = str(class_name).casefold() == "spindle"
            colour = (60, 200, 255) if is_spindle else (255, 60, 60)

            label = f"{class_name} {float(confidence):.2f}"
            label_y = max(0, y1 - 16)
            text_box = draw.textbbox((x1, label_y), label, font=font)

            draw.rectangle((x1, y1, x2, y2), outline=colour, width=3)
            draw.rectangle(
                (
                    text_box[0] - 2,
                    text_box[1] - 2,
                    text_box[2] + 2,
                    text_box[3] + 2,
                ),
                fill=colour,
            )
            draw.text(
                (x1, label_y),
                label,
                fill=(255, 255, 255),
                font=font,
            )

        status = (
            f"RF-DETR M | {camera_name} | objects: {object_count} | "
            f"inference: {latency_ms:.1f} ms"
        )
        status_box = draw.textbbox((8, 8), status, font=font)
        draw.rectangle(
            (4, 4, status_box[2] + 12, status_box[3] + 12),
            fill=(0, 0, 0),
        )
        draw.text((8, 8), status, fill=(255, 255, 255), font=font)

        return np.asarray(image), latency_ms, object_count, payload


class RFDETRProcessedVideoTrack(MediaStreamTrack):
    kind = "video"

    def __init__(
        self,
        source_buffer: LatestFrameBuffer,
        engine: RFDETRStreamingEngine,
        stats: CameraPipelineStats,
        reporter: Optional["DetectionReporter"] = None,
    ):
        super().__init__()
        self.source_buffer = source_buffer
        self.engine = engine
        self.stats = stats
        self.reporter = reporter
        self._last_sequence = 0

    async def recv(self) -> av.VideoFrame:
        sequence, source_frame = await self.source_buffer.newest_after(
            self._last_sequence
        )
        self._last_sequence = sequence

        source_rgb = source_frame.to_ndarray(format="rgb24")
        annotated_rgb, latency_ms, object_count, payload = await asyncio.to_thread(
            self.engine.process,
            source_rgb,
            self.stats.camera_name,
        )

        self.stats.processed_frames += 1
        self.stats.last_inference_ms = latency_ms
        self.stats.last_object_count = object_count

        # Buffered, never awaited: reporting must not be able to stall the
        # video track if Next.js is slow or unreachable.
        if self.reporter is not None:
            self.reporter.push(time.time() * 1000.0, latency_ms, payload)

        output_frame = av.VideoFrame.from_ndarray(annotated_rgb, format="rgb24")
        output_frame.pts = source_frame.pts
        output_frame.time_base = source_frame.time_base
        return output_frame


### 6b. Reporting detections back to Next.js

One `DetectionReporter` per camera batches inferred frames and POSTs them to `/api/inference/detections`.

Two properties matter more than throughput here:

- **It can never stall the video track.** `push()` only appends to a bounded `deque`; all network I/O happens on a separate task. A Next.js outage degrades counting, but the annotated stream keeps flowing.
- **It drops rather than queues.** The deque is capped, so an outage cannot grow memory without bound. Dropping the oldest frames is the right loss: the server takes `max()` over a 2 s window, so frames from an outage several seconds ago would be discarded anyway.


In [ ]:
class DetectionReporter:
    """Batches per-frame detections and ships them to Next.js.

    Deliberately fire-and-forget: every failure path logs and drops. Counting
    degrading is far preferable to the annotated video stream stalling.
    """

    def __init__(
        self,
        camera_id: str,
        base_url: str,
        headers: dict[str, str],
        flush_interval: float = REPORT_FLUSH_SECONDS,
        maxlen: int = REPORT_BUFFER_MAXLEN,
    ):
        self.camera_id = camera_id
        self.url = f"{base_url}/api/inference/detections"
        self.headers = headers
        self.flush_interval = flush_interval
        self._buffer: deque[dict[str, Any]] = deque(maxlen=maxlen)
        self._task: Optional[asyncio.Task[None]] = None
        self._session: Optional[aiohttp.ClientSession] = None
        self.frames_sent = 0
        self.frames_dropped = 0
        self.failures = 0
        self.last_error: Optional[str] = None

    def push(
        self,
        ts_ms: float,
        inference_ms: float,
        detections: list[dict[str, Any]],
    ) -> None:
        """Called from the video track. Must stay synchronous and cheap."""
        if len(self._buffer) == self._buffer.maxlen:
            # deque silently evicts the oldest; count it so the loss is visible
            # in the stats cell rather than being invisible.
            self.frames_dropped += 1
        self._buffer.append(
            {
                "ts": round(ts_ms, 1),
                "inferenceMs": round(inference_ms, 2),
                "detections": detections,
            }
        )

    async def start(self) -> None:
        timeout = aiohttp.ClientTimeout(total=10)
        self._session = aiohttp.ClientSession(timeout=timeout)
        self._task = asyncio.create_task(self._loop())

    async def _loop(self) -> None:
        try:
            while True:
                await asyncio.sleep(self.flush_interval)
                await self._flush()
        except asyncio.CancelledError:
            # Best-effort final flush so the last visit is not lost on shutdown.
            await self._flush()
            raise

    async def _flush(self) -> None:
        if not self._buffer or self._session is None:
            return

        frames = list(self._buffer)
        self._buffer.clear()

        try:
            async with self._session.post(
                self.url,
                json={"cameraId": self.camera_id, "frames": frames},
                headers=self.headers,
            ) as response:
                if response.status >= 300:
                    body = await response.text()
                    self.failures += 1
                    self.last_error = f"HTTP {response.status}: {body[:200]}"
                    if self.failures <= 3 or self.failures % 20 == 0:
                        print(f"[{self.camera_id}] report failed — {self.last_error}")
                else:
                    self.frames_sent += len(frames)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            self.failures += 1
            self.last_error = repr(exc)
            if self.failures <= 3 or self.failures % 20 == 0:
                print(f"[{self.camera_id}] report failed — {self.last_error}")

    async def close(self) -> None:
        if self._task is not None:
            self._task.cancel()
            try:
                await self._task
            except BaseException:
                pass
            self._task = None
        if self._session is not None:
            await self._session.close()
            self._session = None


async def register_processed_sessions(
    sessions: dict[str, dict[str, str]],
) -> None:
    """Tell Next.js which Cloudflare session carries each camera's annotated
    track. Each camera has its own session (see publish_cloudflare_video's
    docstring for why) so the payload is keyed per camera rather than
    carrying one shared sessionId:

        {"role": "processed",
         "sessions": {"CAM-01": {"sessionId": "...", "trackName": "..."}}}
    """
    url = f"{NEXTJS_BASE_URL}/api/inference/register"
    payload = {"role": "processed", "sessions": sessions}
    timeout = aiohttp.ClientTimeout(total=10)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.post(url, json=payload, headers=INFERENCE_HEADERS) as response:
            if response.status >= 300:
                body = await response.text()
                raise RuntimeError(f"HTTP {response.status} from {url}: {body[:300]}")


async def processed_heartbeat_loop(
    sessions: dict[str, dict[str, str]],
    interval_seconds: float = 15.0,
) -> None:
    """Re-register on an interval so Next.js can tell a dead pipeline from a
    quiet one, and mark the dashboard's inference indicator offline if this
    notebook stops."""
    while True:
        try:
            await asyncio.sleep(interval_seconds)
            await register_processed_sessions(sessions)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            print(f"Processed heartbeat failed: {exc!r}")


## 7. Subscribe to every source track and publish all RF-DETR outputs

In [ ]:
async def subscribe_to_cloudflare_video(
    app_id: str,
    app_secret: str,
    source_session_id: str,
    source_track_name: str,
) -> tuple[RTCPeerConnection, CloudflareRealtimeAPI, MediaStreamTrack]:
    api = CloudflareRealtimeAPI(app_id, app_secret)
    pc = RTCPeerConnection(rtc_configuration())
    loop = asyncio.get_running_loop()
    video_future: asyncio.Future[MediaStreamTrack] = loop.create_future()

    @pc.on("track")
    def _on_track(track: MediaStreamTrack) -> None:
        print(
            f"Received source track {source_track_name}: "
            f"kind={track.kind}, id={track.id}"
        )
        if track.kind == "video" and not video_future.done():
            video_future.set_result(track)

    # Negotiate a receive-only VP8 video m-line from the beginning.
    receive_transceiver = pc.addTransceiver("video", direction="recvonly")
    force_vp8(receive_transceiver)

    offer = await pc.createOffer()
    local_offer = await set_complete_local_description(
        pc, offer, stage=f"subscriber {source_track_name}: initial offer"
    )
    session_result = await api.new_session(local_offer.sdp)
    await pc.setRemoteDescription(description_from_result(session_result))
    await wait_for_connection(
        pc, stage=f"subscriber {source_track_name}: initial connection"
    )

    pull_result = await api.new_tracks(
        [
            {
                "location": "remote",
                "sessionId": source_session_id,
                "trackName": source_track_name,
            }
        ]
    )

    if pull_result.get("requiresImmediateRenegotiation"):
        description = pull_result.get("sessionDescription")
        if not description or description.get("type") != "offer":
            raise RuntimeError(
                "Cloudflare requested renegotiation but did not return an offer."
            )
        await pc.setRemoteDescription(
            RTCSessionDescription(
                sdp=description["sdp"],
                type=description["type"],
            )
        )
        force_all_video_transceivers_to_vp8(pc)
        answer = await pc.createAnswer()
        local_answer = await set_complete_local_description(
            pc,
            answer,
            stage=f"subscriber {source_track_name}: renegotiation answer",
        )
        await api.renegotiate(local_answer.sdp)
    elif pull_result.get("sessionDescription"):
        await pc.setRemoteDescription(description_from_result(pull_result))

    try:
        remote_video = await asyncio.wait_for(video_future, timeout=60.0)
    except TimeoutError as exc:
        raise TimeoutError(
            f"Cloudflare did not deliver source video track "
            f"{source_session_id} / {source_track_name} within 60s."
        ) from exc

    return pc, api, remote_video


def slugify_track_part(value: str) -> str:
    slug = re.sub(r"[^a-zA-Z0-9_-]+", "-", value.strip()).strip("-").lower()
    return slug or "camera"


async def publish_cloudflare_video(
    app_id: str,
    app_secret: str,
    track: MediaStreamTrack,
    track_name: str,
) -> tuple[RTCPeerConnection, CloudflareRealtimeAPI, str]:
    """Publish ONE track to its own, independent Cloudflare session.

    Each camera gets its own RTCPeerConnection rather than sharing one
    connection with multiple tracks. Bundling multiple tracks onto one
    connection does not work against Cloudflare Realtime: the offer declares
    a=group:BUNDLE, but the underlying WebRTC stack still negotiates a fully
    independent ICE session per track (distinct ice-ufrag/pwd/candidates)
    rather than sharing one transport the way a browser would, and
    Cloudflare's answer only ever sets up a real ICE session for the first
    track, leaving every additional one referencing a session Cloudflare
    never created. Confirmed directly against this project's edge worker:
    two tracks on one connection never completed ICE; the identical code
    with a single track connected within two seconds. See
    services/edge/main.py's CameraPublisher docstring for the full story.
    """
    api = CloudflareRealtimeAPI(app_id, app_secret)
    pc = RTCPeerConnection(rtc_configuration())

    transceiver = pc.addTransceiver(track, direction="sendonly")
    force_vp8(transceiver)

    offer = await pc.createOffer()
    local_offer = await set_complete_local_description(
        pc, offer, stage=f"processed publisher {track_name}: initial offer"
    )
    session_result = await api.new_session(local_offer.sdp)
    await pc.setRemoteDescription(description_from_result(session_result))

    if transceiver.mid is None:
        raise RuntimeError(f"Cloudflare did not assign a media MID for {track_name}.")

    # Match the working browser sequence: register the local track, then wait
    # for the PeerConnection to reach connected/completed.
    registration_offer = await pc.createOffer()
    local_registration_offer = await set_complete_local_description(
        pc,
        registration_offer,
        stage=f"processed publisher {track_name}: track registration offer",
    )
    tracks_result = await api.new_tracks(
        [{"location": "local", "mid": transceiver.mid, "trackName": track_name}],
        offer_sdp=local_registration_offer.sdp,
    )
    await pc.setRemoteDescription(description_from_result(tracks_result))
    await wait_for_connection(
        pc,
        timeout_seconds=90.0,
        stage=f"processed publisher {track_name} connection",
    )

    if not api.session_id:
        raise RuntimeError(f"Processed publisher session ID is missing for {track_name}.")
    return pc, api, api.session_id


async def start_multi_camera_pipeline() -> dict[str, Any]:
    engine = RFDETRStreamingEngine(
        model=rf_model,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        target_class_names=NORMALIZED_TARGET_CLASS_NAMES,
        inference_shape=INFERENCE_SHAPE,
    )

    run_suffix = uuid.uuid4().hex[:8]
    camera_pipelines: list[dict[str, Any]] = []
    heartbeat_task: Optional[asyncio.Task[None]] = None

    try:
        for index, camera in enumerate(SOURCE_CAMERAS, start=1):
            camera_id = camera["cameraId"]
            name = camera["name"]
            source_session_id = camera["sessionId"]
            source_track_name = camera["trackName"]
            output_track_name = (
                f"rfdetr-m-{slugify_track_part(camera_id)}-{run_suffix}"
            )

            print(
                f"[{index}/{len(SOURCE_CAMERAS)}] Subscribing to "
                f"{camera_id} / {source_track_name}..."
            )
            subscriber_pc, subscriber_api, source_track = (
                await subscribe_to_cloudflare_video(
                    APP_ID,
                    APP_SECRET,
                    source_session_id,
                    source_track_name,
                )
            )

            stats = CameraPipelineStats(
                camera_id=camera_id,
                camera_name=name,
                source_track_name=source_track_name,
                output_track_name=output_track_name,
                started_at=time.monotonic(),
            )
            frame_buffer = LatestFrameBuffer(source_track, stats)
            print(f"Waiting for first decodable VP8 frame: {camera_id}...")
            await frame_buffer.wait_for_first_frame(timeout_seconds=60.0)
            print(f"First frame received: {camera_id}")

            # Keyed on the canonical camera id, which is what the server's
            # entry/exit pairing is configured against.
            reporter = DetectionReporter(
                camera_id=camera_id,
                base_url=NEXTJS_BASE_URL,
                headers=INFERENCE_HEADERS,
            )
            await reporter.start()

            processed_track = RFDETRProcessedVideoTrack(
                source_buffer=frame_buffer,
                engine=engine,
                stats=stats,
                reporter=reporter,
            )

            camera_pipelines.append(
                {
                    "camera": camera,
                    "camera_id": camera_id,
                    "subscriber_pc": subscriber_pc,
                    "subscriber_api": subscriber_api,
                    "source_track": source_track,
                    "frame_buffer": frame_buffer,
                    "processed_track": processed_track,
                    "reporter": reporter,
                    "stats": stats,
                    "output_track_name": output_track_name,
                    "publisher_pc": None,
                }
            )
            print(f"Connected: {camera_id}")

        print(
            f"Publishing {len(camera_pipelines)} processed RF-DETR Medium tracks "
            "(one Cloudflare session per camera)..."
        )
        publish_results = await asyncio.gather(
            *(
                publish_cloudflare_video(
                    APP_ID,
                    APP_SECRET,
                    item["processed_track"],
                    item["output_track_name"],
                )
                for item in camera_pipelines
            )
        )
        for item, (publisher_pc, _publisher_api, session_id) in zip(
            camera_pipelines, publish_results
        ):
            item["publisher_pc"] = publisher_pc
            item["processed_session_id"] = session_id

        processed_sessions = {
            item["camera_id"]: {
                "sessionId": item["processed_session_id"],
                "trackName": item["output_track_name"],
            }
            for item in camera_pipelines
        }

        # Registering is what makes the dashboard show the annotated video; a
        # failure here is worth surfacing rather than swallowing.
        await register_processed_sessions(processed_sessions)
        heartbeat_task = asyncio.create_task(
            processed_heartbeat_loop(processed_sessions)
        )
        print(f"Registered processed sessions with {NEXTJS_BASE_URL}")

        pipeline = {
            "engine": engine,
            "camera_pipelines": camera_pipelines,
            "heartbeat_task": heartbeat_task,
            "processed_sessions": processed_sessions,
        }

        print("\n" + "=" * 88)
        print("PIPELINE RUNNING")
        print("=" * 88)
        for camera_id, entry in processed_sessions.items():
            print(f"  {camera_id} -> session {entry['sessionId']} / track {entry['trackName']}")
        print(
            "\nThe dashboard discovers these automatically at /cameras.\n"
            "Counts appear once a spindle has passed both cameras."
        )
        print("=" * 88)
        return pipeline

    except BaseException:
        if heartbeat_task is not None:
            heartbeat_task.cancel()
        for item in camera_pipelines:
            await item["reporter"].close()
            await item["frame_buffer"].close()
            await item["subscriber_pc"].close()
            publisher_pc = item.get("publisher_pc")
            if publisher_pc is not None:
                await publisher_pc.close()
        raise


async def stop_multi_camera_pipeline(
    pipeline: Optional[dict[str, Any]],
) -> None:
    if not pipeline:
        return

    heartbeat_task = pipeline.get("heartbeat_task")
    if heartbeat_task is not None:
        heartbeat_task.cancel()
        try:
            await heartbeat_task
        except BaseException:
            pass

    for item in pipeline.get("camera_pipelines", []):
        reporter = item.get("reporter")
        if reporter is not None:
            # Closing flushes whatever is still buffered before tearing down.
            await reporter.close()
        frame_buffer = item.get("frame_buffer")
        if frame_buffer is not None:
            await frame_buffer.close()
        subscriber_pc = item.get("subscriber_pc")
        if subscriber_pc is not None:
            await subscriber_pc.close()
        publisher_pc = item.get("publisher_pc")
        if publisher_pc is not None:
            await publisher_pc.close()

    print("Multi-camera pipeline stopped.")


## 8. Start the multi-camera pipeline

Prerequisites, in order:

1. `docker compose up -d edge-worker nextjs` — the edge worker publishes the cameras and registers its Cloudflare session.
2. `cloudflared tunnel --url http://localhost:3000` — and `NEXTJS_BASE_URL` above set to the resulting URL.
3. A production session started on the dashboard, if you want passes persisted. Without one the pipeline still runs and live counts still appear; only the database writes are skipped.

Starting the pipeline subscribes to every source track, publishes one annotated track per camera, registers the processed session with Next.js, and begins streaming detections back for sampling.

If rerunning after a previous start, run the cleanup cell first.


### Codec requirement

The edge worker publishes VP8, and this notebook restricts both its subscriber and its processed publisher transceivers to VP8. This avoids the repeated H.264 decode warnings that can stop the first source frame from ever reaching RF-DETR.

Startup reports the exact stage that timed out, so a failure names the step that broke rather than just the total elapsed time.


In [ ]:
try:
    if PIPELINE is not None:
        await stop_multi_camera_pipeline(PIPELINE)
except NameError:
    pass

PIPELINE = await start_multi_camera_pipeline()


## 9. Inspect all camera pipelines

`frames_reported` should climb steadily. If `report_failures` is rising instead, the counts never reach Next.js — check `last_report_error`. If `frames_dropped` is rising, frames are being produced faster than they can be shipped, which usually means the tunnel is down rather than that the GPU is too fast.


In [ ]:
if not PIPELINE:
    raise RuntimeError("Start the pipeline first.")

print("Processed sessionId:", PIPELINE["result_session_id"])
print("Publisher ICE state:", PIPELINE["publisher_pc"].iceConnectionState)
print()

for index, item in enumerate(PIPELINE["camera_pipelines"], start=1):
    stats = item["stats"]
    reporter = item["reporter"]
    print(
        {
            "index": index,
            "camera_id": stats.camera_id,
            "source_track": stats.source_track_name,
            "processed_track": stats.output_track_name,
            "received_frames": stats.received_frames,
            "processed_frames": stats.processed_frames,
            "processed_fps_average": round(stats.processed_fps, 3),
            "last_inference_ms": round(stats.last_inference_ms, 1),
            "last_object_count": stats.last_object_count,
            "subscriber_ice_state": item["subscriber_pc"].iceConnectionState,
            # Reporting health. frames_dropped rising means Next.js cannot keep
            # up or is unreachable; failures shows why.
            "frames_reported": reporter.frames_sent,
            "frames_dropped": reporter.frames_dropped,
            "report_failures": reporter.failures,
            "last_report_error": reporter.last_error,
        }
    )


## 10. Stop and clean up

Run this before changing the source coordinates, model, or camera set.


In [ ]:
await stop_multi_camera_pipeline(PIPELINE)
PIPELINE = None


## Troubleshooting

**Next.js rejects the API key (HTTP 401)**

- `INFERENCE_API_KEY` here must match the value in the project's `.env`.
- Restart the `nextjs` container after changing `.env`; the key is read at startup.

**No source sessions are registered (HTTP 404 from `/api/inference/source`)**

- Start the edge worker: `docker compose up -d edge-worker`.
- Check its logs for `Source sessions registered`. If it says `INFERENCE_API_KEY is not set`, add it to the edge worker's environment in `docker-compose.yml`.

**A registered source session is stale**

- The edge worker registers on a 15 s heartbeat and Next.js marks each camera's session stale independently after 45 s. A stale entry almost always means the worker died and that camera's Cloudflare session is already gone.

**Entry and exit counts never appear even though video streams fine**

- Each camera publishes on its own Cloudflare session — this is required (see the `publish_cloudflare_video` docstring in section 7). Confirm `/api/inference/live` shows a distinct `sessionId` per camera under `processedSessions`; if two cameras show the same id, an old build is running.

**Cannot reach Next.js at all**

- Confirm `cloudflared tunnel --url http://localhost:3000` is still running; its URL changes every restart.
- Open `NEXTJS_BASE_URL` in a browser to confirm the tunnel is up before blaming the notebook.

**Video streams but counts never appear on the dashboard**

- Check `report_failures` and `frames_dropped` in the stats cell.
- Counts require a spindle to pass **both** cameras — a single camera's visit stays queued until its pair arrives. `queueDepth` on the dashboard shows how many are waiting.
- `spindle_pass` rows are only written while a production session is active. Start one on the dashboard.

**Counts appear but look wrong**

- Watch the annotated stream: the cyan box is the spindle. Toys are counted only if their centroid falls inside it, plus `SPINDLE_BOUNDARY_MARGIN`.
- Every threshold is server-side. Adjust `SPINDLE_BOUNDARY_MARGIN`, `SPINDLE_MIN_CONFIDENCE`, `HOTWHEELS_MIN_CONFIDENCE`, `MAX_HOTWHEELS`, or `DETECTION_INTERVAL_MS` in `.env`, restart `nextjs`, and leave this notebook running.
- `DETECTION_INTERVAL_MS` must span at least one full spindle rotation. The whole `max()` premise is that some frame in the window catches the spindle at an angle where no toy is hidden behind the post.

**Entry and exit counts are consistently offset by one spindle**

- One physical spindle must produce exactly one visit per camera. If a spindle's detection flickers badly it can split into two visits and shift every later pairing. Raise `SPINDLE_ABSENT_INTERVALS` so a longer gap is required before a visit closes.

**Checkpoint not found / cannot be inferred**

- Confirm Google Drive is mounted and use the full `/content/drive/MyDrive/...` path.
- The loader falls back to `RFDETRMedium(pretrain_weights=...)` for older Medium checkpoints.
- Set `TRUST_CHECKPOINT=True` only for a checkpoint you created or fully trust.

**Incorrect or missing class names**

- Check `MODEL_CLASS_NAMES` after the loading cell.
- Class names are matched by name server-side, not by index, so retraining can safely reorder them — but a *renamed* class must be added to `SPINDLE_CLASSES` / `HOTWHEELS_CLASSES` in `lib/inference/constants.ts`.

**CUDA out of memory**

- Reduce the number of active cameras, or stop old pipelines before starting a new one.
- Keep `OPTIMIZE_COMPILE=False` and `INFERENCE_SHAPE=None`.

**Latency grows with more cameras**

- One model is shared and inference is serialized for safety. Each camera drops stale frames so no backlog builds, but per-camera FPS falls as camera count rises.
- Raise `CONFIDENCE`, lower `MAX_DETECTIONS`, or filter `TARGET_CLASS_NAMES`.
